# 20. Model Comparison Report

Compare three ways to classify the same CIFAR-10 images:

1. **Traditional ML:** logistic regression on downsampled RGB pixels.
2. **CNN from scratch:** a small convolutional network trained on CIFAR-10.
3. **Transfer learning:** the saved CIFAR-10 ResNet-18 from Notebook 10.

We compare predictive metrics and practical trade-offs. The goal is not just to crown the highest score, but to choose a model that fits its use case.

## 1. Comparison plan and fairness notes

**Problem:** classify CIFAR-10 images into one of ten classes.

**Evaluation:** use the same deterministic, balanced set of 1,000 official CIFAR-10 test images for all three models. Report accuracy and **macro** precision, recall, and F1 so each class contributes equally. Training uses 400 examples per class for the traditional model and scratch CNN. The existing ResNet was trained earlier on a different 5,000-image subset and ImageNet pretraining, so its training data and compute budget are not identical.

**Important limitation:** Earlier course notebooks also evaluated on the official CIFAR-10 test set. This report is therefore an instructional comparison, not a fresh independent benchmark. The transfer model's original training duration is taken from Notebook 10's recorded CPU epoch timings; its checkpoint is loaded here rather than retrained.

## 2. Load data and define the comparison split

**Decision:** Load only the local CIFAR-10 files, with downloading disabled. Fix the random seed and select a balanced training subset and balanced test subset so the comparison is repeatable.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
import time
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms
SEED = 42
DATA_ROOT = Path("data")
CHECKPOINT = Path("artifacts/resnet18_cifar10_finetuned.pth")
BATCH_SIZE = 128
EPOCHS = 2
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
if not (DATA_ROOT / "cifar-10-batches-py").is_dir():
    raise FileNotFoundError("Local CIFAR-10 data is missing from data/.")
if not CHECKPOINT.is_file():
    raise FileNotFoundError("Run Notebook 10 first to create the ResNet-18 checkpoint.")
raw_train = datasets.CIFAR10(DATA_ROOT, train=True, download=False)
raw_test = datasets.CIFAR10(DATA_ROOT, train=False, download=False)
class_names = raw_train.classes
train_ids, test_ids = [], []
for class_id in range(len(class_names)):
    train_class_ids = np.flatnonzero(np.asarray(raw_train.targets) == class_id)
    test_class_ids = np.flatnonzero(np.asarray(raw_test.targets) == class_id)
    rng.shuffle(train_class_ids)
    rng.shuffle(test_class_ids)
    train_ids.extend(train_class_ids[:400])
    test_ids.extend(test_class_ids[:100])
y_train = np.asarray(raw_train.targets)[train_ids]
y_test = np.asarray(raw_test.targets)[test_ids]
assert len(train_ids) == 4000 and len(test_ids) == 1000
print("Device:", device)
print("Classes:", class_names)
print("Training / comparison test images:", len(train_ids), "/", len(test_ids))
print("Test examples per class:", np.bincount(y_test, minlength=len(class_names)))

Device: cpu
Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
Training / comparison test images: 4000 / 1000
Test examples per class: [100 100 100 100 100 100 100 100 100 100]


## 3. Approach A — Traditional ML: pixel features + logistic regression

**Idea:** Resize each RGB image to 16 × 16 and flatten its pixels into a vector. A multinomial logistic-regression classifier maps that vector to one of ten class scores.

**Why include it?** It is a simple, fast-to-train reference that uses no convolutional feature learning. Flattened pixels discard some spatial structure, so this baseline may struggle with changes in object position or appearance.

**Training time** includes resizing and fitting. No image augmentation is used for this traditional model.

In [2]:
def pixel_features(images):
    features = []
    for image in images:
        small = Image.fromarray(image).resize((16, 16), Image.Resampling.BILINEAR)
        features.append(np.asarray(small, dtype=np.float32).reshape(-1) / 255.0)
    return np.stack(features)
start = time.perf_counter()
X_train_pixels = pixel_features(raw_train.data[train_ids])
X_test_pixels = pixel_features(raw_test.data[test_ids])
traditional_model = LogisticRegression(max_iter=500, tol=0.001, random_state=SEED)
traditional_model.fit(X_train_pixels, y_train)
traditional_training_seconds = time.perf_counter() - start
print("Feature shape:", X_train_pixels.shape)
print(f"Feature extraction + training time: {traditional_training_seconds:.2f} seconds")

Feature shape: (4000, 768)
Feature extraction + training time: 3.06 seconds


**Interpretation:** Each image is represented by 768 pixel values (`16 × 16 × 3`). The classifier learns a set of class weights over those values. Training is straightforward, but these raw pixel features do not explicitly learn edges, shapes, or object parts.

## 4. Approach B — CNN trained from scratch

**Idea:** A small CNN learns visual features and a classifier together from randomly initialized weights. Training uses the same 4,000 training images and a random horizontal flip; the test images are not augmented.

**Decision:** Use two short epochs to keep this comparison notebook manageable. This is a demonstration of training time and trade-offs, not a search for the best possible CNN.

In [3]:
cifar_mean = (0.4914, 0.4822, 0.4465)
cifar_std = (0.2470, 0.2435, 0.2616)
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(), transforms.ToTensor(),
    transforms.Normalize(cifar_mean, cifar_std),
])
eval_transform = transforms.Compose([
    transforms.ToTensor(), transforms.Normalize(cifar_mean, cifar_std),
])
cnn_train_data = datasets.CIFAR10(DATA_ROOT, train=True, download=False, transform=train_transform)
cnn_test_data = datasets.CIFAR10(DATA_ROOT, train=False, download=False, transform=eval_transform)
cnn_train_loader = DataLoader(Subset(cnn_train_data, train_ids), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
cnn_test_loader = DataLoader(Subset(cnn_test_data, test_ids), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Linear(32 * 8 * 8, 10)

    def forward(self, images):
        return self.classifier(self.features(images).flatten(1))
cnn_model = SmallCNN().to(device)
cnn_optimizer = torch.optim.Adam(cnn_model.parameters(), lr=0.003)
loss_fn = nn.CrossEntropyLoss()
if device.type == "cuda":
    torch.cuda.synchronize()
start = time.perf_counter()
for epoch in range(EPOCHS):
    cnn_model.train()
    for images, labels in cnn_train_loader:
        images, labels = images.to(device), labels.to(device)
        cnn_optimizer.zero_grad(set_to_none=True)
        loss_fn(cnn_model(images), labels).backward()
        cnn_optimizer.step()
    print(f"Completed scratch CNN epoch {epoch + 1}/{EPOCHS}")
if device.type == "cuda":
    torch.cuda.synchronize()
cnn_training_seconds = time.perf_counter() - start
print(f"Scratch CNN training time: {cnn_training_seconds:.2f} seconds")

Completed scratch CNN epoch 1/2
Completed scratch CNN epoch 2/2
Scratch CNN training time: 6.61 seconds


**Interpretation:** The CNN learns its visual features from this small CIFAR-10 subset. With only two epochs and 4,000 images, it may be less accurate than a pretrained model, but it requires no external pretraining data or pretrained weights.

## 5. Approach C — Transfer learning: saved ResNet-18

**Idea:** Reuse the fine-tuned ResNet-18 checkpoint produced in Notebook 10. It began with ImageNet-pretrained weights and was adapted to CIFAR-10. Here we load it for inference instead of retraining it.

**Training-time note:** Notebook 10 recorded CPU epoch times of 62.2 s + 57.9 s for classifier training and 88.9 s for fine-tuning (about 209 s total, excluding setup and evaluation). This historical time is not a fresh run in this notebook and is not directly comparable to the two-epoch scratch or traditional-model timings.

In [4]:
transfer_transform = transforms.Compose([
    transforms.Resize((96, 96)), transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])
transfer_test_data = datasets.CIFAR10(DATA_ROOT, train=False, download=False, transform=transfer_transform)
transfer_test_loader = DataLoader(Subset(transfer_test_data, test_ids), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
transfer_model = models.resnet18(weights=None)
transfer_model.fc = nn.Linear(transfer_model.fc.in_features, len(class_names))
transfer_model.load_state_dict(torch.load(CHECKPOINT, map_location="cpu", weights_only=True))
transfer_model = transfer_model.to(device).eval()
transfer_training_seconds = 62.2 + 57.9 + 88.9
print("Loaded:", CHECKPOINT)
print(f"Notebook 10 recorded training time: {transfer_training_seconds:.1f} seconds (historical)")

Loaded: artifacts\resnet18_cifar10_finetuned.pth
Notebook 10 recorded training time: 209.0 seconds (historical)


**Interpretation:** Transfer learning starts with useful visual features learned elsewhere, which can improve performance when the target dataset is modest. The trade-off is a larger, more complex model and dependence on pretrained weights. Notebook 10's reported training duration is a historical reference, not a controlled timing for this experiment.

## 6. Compare accuracy, precision, recall, and F1

**Decision:** Calculate all four metrics on the same 1,000 test images. Use macro precision, macro recall, and macro F1 so every class has equal weight. Accuracy is the share of all predictions that are correct.

In [6]:
def timed_predictions(model, loader):
    model.eval()
    predictions, labels_all = [], []
    if device.type == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.inference_mode():
        for images, labels in loader:
            predictions.extend(model(images.to(device)).argmax(1).cpu().numpy())
            labels_all.extend(labels.numpy())
    if device.type == "cuda":
        torch.cuda.synchronize()
    seconds = time.perf_counter() - start
    return np.asarray(labels_all), np.asarray(predictions), seconds

def metric_row(name, actual, predicted):
    return {
        "Approach": name,
        "Accuracy": accuracy_score(actual, predicted),
        "Precision (macro)": precision_score(actual, predicted, average="macro", zero_division=0),
        "Recall (macro)": recall_score(actual, predicted, average="macro", zero_division=0),
        "F1 (macro)": f1_score(actual, predicted, average="macro", zero_division=0),
    }

start = time.perf_counter()
traditional_predictions = traditional_model.predict(X_test_pixels)
traditional_inference_seconds = time.perf_counter() - start
cnn_actual, cnn_predictions, cnn_inference_seconds = timed_predictions(cnn_model, cnn_test_loader)
transfer_actual, transfer_predictions, transfer_inference_seconds = timed_predictions(transfer_model, transfer_test_loader)
assert np.array_equal(cnn_actual, y_test) and np.array_equal(transfer_actual, y_test)
metric_table = pd.DataFrame([
    metric_row("Traditional: pixel logistic regression", y_test, traditional_predictions),
    metric_row("CNN: trained from scratch", y_test, cnn_predictions),
    metric_row("Transfer: ResNet-18", y_test, transfer_predictions),
])
print(metric_table.round(3).to_string(index=False))

                              Approach  Accuracy  Precision (macro)  Recall (macro)  F1 (macro)
Traditional: pixel logistic regression     0.356              0.355           0.356       0.355
             CNN: trained from scratch     0.418              0.453           0.418       0.409
                   Transfer: ResNet-18     0.839              0.844           0.839       0.840


**Interpretation:** Higher scores indicate stronger classification on this particular test subset. Precision, recall, and F1 help reveal different error patterns; no single metric explains deployment suitability. The subset and test-set reuse caveats still apply.

## 7. Compare parameter count, training time, inference time, and model size

**Explanation:** Parameter count and file size describe different aspects of model footprint. Inference time below is measured over the same test subset, divided by its image count; model preprocessing and data loading are excluded. Classical-model parameter count is the number of learned logistic-regression coefficients and intercepts. These counts are useful descriptions, but not perfectly equivalent across model types.

In [7]:
traditional_parameters = traditional_model.coef_.size + traditional_model.intercept_.size
cnn_parameters = sum(parameter.numel() for parameter in cnn_model.parameters())
transfer_parameters = sum(parameter.numel() for parameter in transfer_model.parameters())
with TemporaryDirectory() as temp_dir:
    temp_dir = Path(temp_dir)
    traditional_file = temp_dir / "traditional.joblib"
    cnn_file = temp_dir / "scratch_cnn.pth"
    joblib.dump(traditional_model, traditional_file)
    torch.save(cnn_model.cpu().state_dict(), cnn_file)
    traditional_size_mb = traditional_file.stat().st_size / (1024 ** 2)
    cnn_size_mb = cnn_file.stat().st_size / (1024 ** 2)
cnn_model = cnn_model.to(device).eval()
transfer_size_mb = CHECKPOINT.stat().st_size / (1024 ** 2)

resource_table = pd.DataFrame([
    {"Approach": "Traditional: pixel logistic regression", "Parameters / coefficients": traditional_parameters,
     "Training time (s)": traditional_training_seconds, "Inference (ms/image)": traditional_inference_seconds * 1000 / len(y_test),
     "Saved model size (MB)": traditional_size_mb},
    {"Approach": "CNN: trained from scratch", "Parameters / coefficients": cnn_parameters,
     "Training time (s)": cnn_training_seconds, "Inference (ms/image)": cnn_inference_seconds * 1000 / len(y_test),
     "Saved model size (MB)": cnn_size_mb},
    {"Approach": "Transfer: ResNet-18", "Parameters / coefficients": transfer_parameters,
     "Training time (s)": transfer_training_seconds, "Inference (ms/image)": transfer_inference_seconds * 1000 / len(y_test),
     "Saved model size (MB)": transfer_size_mb},
])
print(resource_table.round(3).to_string(index=False))

                              Approach  Parameters / coefficients  Training time (s)  Inference (ms/image)  Saved model size (MB)
Traditional: pixel logistic regression                       7690              3.063                 0.041                  0.030
             CNN: trained from scratch                      25578              6.606                 0.572                  0.101
                   Transfer: ResNet-18                   11181642            209.000                 7.363                 42.731


**Interpretation:** These numbers describe this machine and these saved formats. Traditional training time includes feature resizing; neural-network training times exclude data loading setup. ResNet training time is historical Notebook 10 timing, not a fresh comparable run. Neural inference is batched and measured over the full 1,000-image subset; the milliseconds per image are an average, not single-request latency. Do not treat small timing differences as universal.

## 8. Complexity, data requirement, and interpretability

These qualities are not captured by accuracy alone. The table is a qualitative summary for the specific approaches used here.

In [8]:
qualitative_table = pd.DataFrame([
    {"Approach": "Traditional: pixel logistic regression", "Implementation complexity": "Low",
     "Target data need": "Low to moderate for a simple baseline", "Interpretability": "Relatively high: inspect class weights, though pixel weights are not intuitive"},
    {"Approach": "CNN: trained from scratch", "Implementation complexity": "Moderate",
     "Target data need": "Usually higher; learns visual features from target examples", "Interpretability": "Low: learned filters and features are not directly human-readable"},
    {"Approach": "Transfer: ResNet-18", "Implementation complexity": "Higher: architecture, preprocessing, checkpoint, and pretrained source matter",
     "Target data need": "Can be lower than training from scratch when pretrained features transfer well", "Interpretability": "Low: a deep model; use diagnostic tools cautiously"},
])
print(qualitative_table.to_string(index=False))

                              Approach                                                     Implementation complexity                                                               Target data need                                                               Interpretability
Traditional: pixel logistic regression                                                                           Low                                          Low to moderate for a simple baseline Relatively high: inspect class weights, though pixel weights are not intuitive
             CNN: trained from scratch                                                                      Moderate                    Usually higher; learns visual features from target examples              Low: learned filters and features are not directly human-readable
                   Transfer: ResNet-18 Higher: architecture, preprocessing, checkpoint, and pretrained source matter Can be lower than training from scratch when pretrained fe

**Interpretation:** The traditional model is simplest and most inspectable, but its raw-pixel representation is limited. A scratch CNN avoids an external pretrained model but needs sufficient target data and training. Transfer learning can deliver stronger results with less target-specific training, at the cost of a larger model and added dependency and deployment complexity.

## 9. Final model choice: reason from the use case

**For this CIFAR-10 learning project, choose the transfer-learning ResNet-18 as the accuracy-focused final model**, provided the deployment can accept its roughly 43 MB checkpoint, ResNet runtime, and framework requirements. It benefits from ImageNet pretraining and, in the earlier project comparison, reached about 84% accuracy on the selected test subset.

That is not a universal winner:

- Choose **traditional logistic regression** when a fast, small, easy-to-explain baseline is valuable and moderate image accuracy is acceptable.
- Choose a **CNN from scratch** when avoiding pretrained weights is important, target data is sufficient, and the team can afford training and validation.
- Choose **transfer learning** when target data is limited and higher predictive performance is worth the extra model size, complexity, and pretrained-weight dependency.

Before a real release, measure end-to-end latency on target hardware, validate against a truly untouched representative test set, and compare model size and error costs with the application's constraints. The best final model is the smallest and simplest approach that meets the required accuracy, reliability, and response-time targets—not automatically the one with the highest score.